# Computer Vision: CNN, аугментации, Transfer Learning

Шаблоны для задач с изображениями: кастомный Dataset, аугментации, своя CNN, fine-tuning pretrained ResNet.

## Импорты + seed

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import f1_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Аугментации

In [ ]:
IMG_SIZE = 224  # 224 для ResNet, 32 для CIFAR-подобных

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Dataset для изображений

In [ ]:
class ImageDataset(Dataset):
    """Универсальный Dataset: принимает список путей + лейблы."""
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.labels is not None:
            return img, self.labels[idx]
        return img

# Пример использования:
# train_ds = ImageDataset(train_paths, train_labels, transform=train_transform)
# val_ds = ImageDataset(val_paths, val_labels, transform=val_transform)
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
# val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# Альтернатива: ImageFolder (если данные лежат в папках по классам)
# train/
#   class_0/
#     img1.jpg
#   class_1/
#     img2.jpg

from torchvision.datasets import ImageFolder

# train_ds = ImageFolder('train/', transform=train_transform)
# val_ds = ImageFolder('val/', transform=val_transform)
# print(f'Classes: {train_ds.classes}')

## Своя CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# model = SimpleCNN(num_classes=10).to(device)

## Transfer Learning: ResNet (рекомендуется)

In [ ]:
NUM_CLASSES = 10  # <-- подставить

# Вариант 1: Feature extraction (быстро, мало данных)
model = models.resnet18(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

# Вариант 2: Full fine-tuning (больше данных, лучше качество)
# model = models.resnet18(weights='IMAGENET1K_V1')
# model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
# model = model.to(device)

# Вариант 3: Частичный fine-tuning (разморозить последние слои)
# model = models.resnet18(weights='IMAGENET1K_V1')
# for param in model.parameters():
#     param.requires_grad = False
# for param in model.layer4.parameters():
#     param.requires_grad = True
# model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
# model = model.to(device)

# Другие backbone'ы:
# model = models.resnet50(weights='IMAGENET1K_V1')  -- тяжелее, точнее
# model = models.efficientnet_b0(weights='IMAGENET1K_V1'); model.classifier[1] = nn.Linear(1280, NUM_CLASSES)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Train / Eval loop для CV

In [ ]:
NUM_EPOCHS = 20
LR = 1e-3  # для feature extraction; 1e-4 для fine-tuning

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_f1 = 0
for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    all_preds, all_labels = [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    scheduler.step()
    train_f1 = f1_score(all_labels, all_preds, average='macro')

    # --- Val ---
    model.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_f1 = f1_score(all_labels, all_preds, average='macro')

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d} | train_f1={train_f1:.4f} | val_f1={val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_cv_model.pt')

print(f'\nBest val F1: {best_f1:.4f}')
model.load_state_dict(torch.load('best_cv_model.pt', map_location=device))

## Предсказание + Submission

In [ ]:
import pandas as pd

model.eval()
test_preds = []
with torch.no_grad():
    for imgs in test_loader:
        if isinstance(imgs, (list, tuple)):
            imgs = imgs[0]
        imgs = imgs.to(device)
        logits = model(imgs)
        test_preds.extend(logits.argmax(1).cpu().numpy())

submission = pd.DataFrame({'id': test_ids, 'label': test_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

## TTA (Test-Time Augmentation) -- олимпиадная хитрушечка


In [ ]:
tta_transforms = [
    val_transform,  # преобразование для оригинальных изображений
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(),
               T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]),
]

model.eval()
all_probs = []
for tta_t in tta_transforms:
    tta_ds = ImageDataset(test_paths, transform=tta_t)
    tta_loader = DataLoader(tta_ds, batch_size=64, shuffle=False)
    probs = []
    with torch.no_grad():
        for imgs in tta_loader:
            if isinstance(imgs, (list, tuple)):
                imgs = imgs[0]
            logits = model(imgs.to(device))
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    all_probs.append(np.vstack(probs))

avg_probs = np.mean(all_probs, axis=0)
tta_preds = avg_probs.argmax(axis=1)